# Folio · understand document RAG, one step at a time

RAG means **retrieve → augment → generate**. We do not train Gemini or Groq on your files. We split files into passages, find relevant passages in Qdrant, and put those passages into a bounded prompt.

This notebook is the starting point. Run top to bottom in the project virtual environment after following README.md. The same small Python functions power the web app. Retrieval works without an LLM key; the final answer needs either `GOOGLE_API_KEY` or `GROQ_API_KEY` in `.env`.

**Flow:** PDF/TXT/Markdown → LangChain Documents → chunks → dense + BM25 vectors → Qdrant hybrid retrieval → 5,000-token application budget → Gemini, then Groq on failure.

The notebook uses a separate in-memory collection, so it never alters your web app's documents. First use downloads the local embedding models.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("FASTEMBED_CACHE_PATH", str(ROOT / "data" / "models"))
from folio.rag import read_document, split_documents, open_store, build_prompt, generate, token_count
from langchain_core.messages import HumanMessage, AIMessage


## 1. Load a document
A LangChain `Document` contains text plus metadata. Filenames and PDF page numbers travel with each passage so answers can show their sources. Supported files: text PDFs, UTF-8 TXT, and Markdown. Scanned PDFs need OCR first.

We start with a tiny sample that has an answer we can verify.

In [ ]:
sample = (ROOT / "examples" / "team-handbook.md").read_bytes()
documents = read_document("team-handbook.md", sample)
print(documents[0].metadata)
print(documents[0].page_content)


## 2. Split into overlapping passages
Long documents cannot all fit in the prompt. Each passage is at most 350 reference tokens with 60 tokens of overlap. Overlap preserves sentences near boundaries. The reference tokenizer is `cl100k_base`, not either provider's native tokenizer.

In [ ]:
chunks = split_documents(documents)
print(f"{len(documents)} documents → {len(chunks)} chunks")
for chunk in chunks:
    print(token_count(chunk.page_content), chunk.page_content[:120])


## 3. Store two representations in Qdrant
**Dense vectors** (`BAAI/bge-small-en-v1.5`, local FastEmbed) capture semantic similarity. **Sparse BM25 vectors** (`Qdrant/bm25`) preserve keyword matching. Qdrant applies IDF weights to sparse vectors and reciprocal rank fusion (RRF) combines the two rankings.

The embedding models run locally and do not depend on Gemini or Groq, so an LLM provider outage does not prevent retrieval. These defaults are English-oriented.

In [ ]:
from qdrant_client import QdrantClient
from uuid import uuid4

client = QdrantClient(":memory:")
store = open_store(client, "notebook_demo")
store.add_documents(chunks, ids=[str(uuid4()) for _ in chunks])
print(client.get_collection("notebook_demo").points_count, "indexed chunks")


## 4. Retrieve with hybrid search
Try a paraphrase (semantic) and an exact policy code (keyword). Hybrid fusion is a ranking, not a probability; we do not invent a universal relevance threshold.

In [ ]:
question = "What is the learning budget, and does a manager need to approve it?"
hits = store.similarity_search(question, k=6)
for i, hit in enumerate(hits, 1):
    print(f"[{i}] {hit.metadata['source']}: {hit.page_content[:240]}")


## 5. Fit everything into a 5,000-token application window
The app counts the **whole prompt** using `cl100k_base`, allows 3,900 input tokens, reserves 1,000 output tokens and 100 for formatting. It includes the system instructions, current question, recent complete conversation turns (up to 700 tokens), and evidence. Old turns are dropped first; evidence is clipped only to the remaining budget. Oversized questions are rejected rather than silently truncated.

This is a repeatable application budget, **not an exact Gemini/Llama native-token limit**. Their tokenizers differ. The output cap is also passed to each provider. Source excerpts returned to the UI are exactly those placed in the prompt.

In [ ]:
history = []  # Example: [HumanMessage("What is policy L-104?"), AIMessage("It covers learning.")]
messages, sources, used_tokens = build_prompt(question, hits, history)
print(f"Prompt: {used_tokens}; reserve: 1100; total: {used_tokens + 1100}/5000")
assert used_tokens + 1100 <= 5000
print(messages[-1].content)


## 6. Generate with automatic provider fallback
Only configured providers are tried. `PRIMARY_PROVIDER` chooses the first one. A timeout, quota error, empty answer, or provider failure moves to the other provider using the same evidence. If both fail, the app reports an error; it never invents a success. Keys remain on the Python server.

Retrieved documents are untrusted data, not instructions. The system prompt asks for evidence-backed answers and bracketed citations, and asks the model to admit when the documents do not answer the question. This reduces hallucination; it cannot guarantee correctness.

In [ ]:
if os.getenv("GOOGLE_API_KEY") or os.getenv("GROQ_API_KEY"):
    answer, provider = generate(messages)
    print(f"Provider: {provider}\n\n{answer}")
else:
    print("Retrieval completed. Add a Gemini or Groq key to .env, then rerun the setup and this cell.")


## 7. Follow-up questions
The web app retains the last conversation turns in this browser tab. Retrieval combines the latest user question with the preceding user question for simple follow-ups such as “Does that need approval?”. The prompt also receives bounded conversation history. There is no extra question-rewriting LLM call.

`src/folio/rag.py` holds this pipeline; `src/folio/app.py` provides three endpoints; `src/folio/static/` is the interface. Run `uv run uvicorn folio.app:app --host 127.0.0.1 --port 8000` from the project root after closing the notebook.

Try changing `question`, loading your own file, or comparing `RetrievalMode.DENSE`, `SPARSE`, and `HYBRID` in a separate store. Run the small checks with `uv run python -m unittest discover -s tests`.

In [ ]:
client.close()
